In [21]:
import torch
import torch.optim as optim
import torch.nn as nn
from hyperframe.frame import DataFrame

from simple_model.model import ProtENN2_style

In [22]:
model_oh = ProtENN2_style(cnn_dim=256,
                          kernel_size=5,
                          dilation=2,
                          in_channels=21,
                          num_pfams=297)

model_oh.load_state_dict(torch.load("./saved_models/random_oh.pt", map_location=torch.device('cpu')))

<All keys matched successfully>

In [23]:
model_emb = ProtENN2_style(cnn_dim=256,
                           kernel_size=5,
                           dilation=2,
                           in_channels=1024,
                           num_pfams=297)

model_emb.load_state_dict(torch.load("./saved_models/random_emb.pt", map_location=torch.device('cpu')))

<All keys matched successfully>

In [24]:
import numpy as np
import pandas as pd

data = pd.read_pickle("../dataset/dataset.pkl")

In [25]:
import h5py

# Load data from emb file
emb_path = "../dataset/test_sequences_emb.h5"

with h5py.File(emb_path, "r") as f:
    keys = list(f.keys())
    # get all embeddings
    embeddings = []
    for key in keys:
        d = f[key][:]
        if isinstance(d, np.ndarray):
            embeddings.append(d)
        else:
            print(f"Data for key {key} is not a numpy array.")

# Makes sure keys are in same order as in emb training
data = data.loc[keys]
data["embedding"] = embeddings

data = data.sample(frac=1, random_state=42).reset_index(drop=True)

In [26]:
data

,pfam_tensor,sequence,embedding
0,"[None, None, None, None, None, None, None, Non...",MDKNLMMPKRSRIDVKGNFANGPLQARPLVALLDGRDCSIEMPILK...,"[[-0.020949217, -0.12391873, 0.0009043915, 0.2..."
1,"[None, None, None, None, None, None, None, Non...",MFRRMAVTSLQKGLSRRAFCNTPRLLNLDYQAYKTATVREAAPEWA...,"[[-0.21183026, -0.012223753, -0.10167508, 0.41..."
2,"[PF00244.26, PF00244.26, PF00244.26, PF00244.2...",MAKLSEQTERFDEMLDYMKKILDMSKELKDEERNLLSIAYKNYVSQ...,"[[0.2829769, -0.09260906, 0.034509134, 0.06597..."
3,"[None, None, None, None, None, None, None, Non...",MAIYTVKALITDPIDEILIKTLREKGIQVDYMPEISKEELLNIIGN...,"[[0.41590053, -0.27867454, 0.107742675, -0.001..."
4,"[None, None, None, None, None, None, None, Non...",MFRALLIVLLVLRCTHATHAQLSYYTNMQNQVMVWDKGMIRKVDYL...,"[[0.104467444, -0.47745505, 0.18655157, 0.0509..."
...,...,...,...
216,"[None, None, None, None, None, None, None, Non...",MEQTENVEAAPAVPEDPEKARLVESARLAEQAERYDDMVENMKKVV...,"[[0.21143903, -0.17015953, -0.13576733, 0.1439..."
217,"[None, None, None, None, None, None, None, Non...",MAEITNDEGKSKEEQKMQMIITAQIAFDADRCDDMADCMHKATVIA...,"[[0.1818518, 0.027461898, -0.022101969, 0.2415..."
218,"[None, None, None, None, None, None, None, Non...",MKSTIMHELKPDMKRDNLLFLARLSQQTERFPEMLEYMKRIIQQPQ...,"[[0.15590605, -0.04418983, -0.28836054, 0.5661..."
219,"[None, None, None, None, None, None, None, Non...",MDELRERNIVLAKLCEQAERYDEMVKAMIEIATNTETELTVEERNL...,"[[-0.038774293, -0.18798219, -0.061563067, 0.2..."


In [27]:
# Generate pfam to index mapping
pfam_to_index = {pfam: idx + 1 for idx, pfam in enumerate(data["pfam_tensor"].explode().unique())}


# Convert pfams to indices and pad to max length
def convert_pfams_to_indices(pfams, pfam_to_index, max_length=1000):
    indices = [pfam_to_index[pfam] for pfam in pfams if pfam in pfam_to_index]
    if len(indices) < max_length:
        indices += [0] * (max_length - len(indices))  # Pad with zeros
    return np.array(indices[:max_length], dtype=np.int64)


data["pfams_indices"] = data["pfam_tensor"].apply(lambda x: convert_pfams_to_indices(x, pfam_to_index))

In [28]:
from simple_model.model_parts import CUSTOM_ALPHABET

MAX_PROTEIN_LENGTH = 1000


def one_hot_encode_sequence(sequence, alphabet=CUSTOM_ALPHABET, max_length=MAX_PROTEIN_LENGTH):
    one_hot = np.zeros((max_length, len(alphabet)), dtype=np.float32)
    for i, char in enumerate(sequence):
        if i < max_length and char in alphabet:
            one_hot[i, alphabet[char]] = 1.0
    return one_hot


data["sequence_oh"] = data["sequence"].apply(one_hot_encode_sequence)

In [29]:
# Pad the embeddings
def pad_embeddings(embedding, max_length=MAX_PROTEIN_LENGTH):
    if len(embedding) < max_length:
        # Pad with zeros if the embedding is shorter than max_length
        padding = np.zeros((max_length - len(embedding), embedding.shape[1]), dtype=np.float32)
        return np.vstack((embedding, padding))
    else:
        # Truncate if the embedding is longer than max_length
        return embedding[:max_length]


data["embedding"] = data["embedding"].apply(pad_embeddings)

In [30]:
data

,pfam_tensor,sequence,embedding,pfams_indices,sequence_oh
0,"[None, None, None, None, None, None, None, Non...",MDKNLMMPKRSRIDVKGNFANGPLQARPLVALLDGRDCSIEMPILK...,"[[-0.020949217, -0.12391873, 0.0009043915, 0.2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,"[None, None, None, None, None, None, None, Non...",MFRRMAVTSLQKGLSRRAFCNTPRLLNLDYQAYKTATVREAAPEWA...,"[[-0.21183026, -0.012223753, -0.10167508, 0.41...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,"[PF00244.26, PF00244.26, PF00244.26, PF00244.2...",MAKLSEQTERFDEMLDYMKKILDMSKELKDEERNLLSIAYKNYVSQ...,"[[0.2829769, -0.09260906, 0.034509134, 0.06597...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,"[None, None, None, None, None, None, None, Non...",MAIYTVKALITDPIDEILIKTLREKGIQVDYMPEISKEELLNIIGN...,"[[0.41590053, -0.27867454, 0.107742675, -0.001...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,"[None, None, None, None, None, None, None, Non...",MFRALLIVLLVLRCTHATHAQLSYYTNMQNQVMVWDKGMIRKVDYL...,"[[0.104467444, -0.47745505, 0.18655157, 0.0509...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...
216,"[None, None, None, None, None, None, None, Non...",MEQTENVEAAPAVPEDPEKARLVESARLAEQAERYDDMVENMKKVV...,"[[0.21143903, -0.17015953, -0.13576733, 0.1439...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
217,"[None, None, None, None, None, None, None, Non...",MAEITNDEGKSKEEQKMQMIITAQIAFDADRCDDMADCMHKATVIA...,"[[0.1818518, 0.027461898, -0.022101969, 0.2415...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
218,"[None, None, None, None, None, None, None, Non...",MKSTIMHELKPDMKRDNLLFLARLSQQTERFPEMLEYMKRIIQQPQ...,"[[0.15590605, -0.04418983, -0.28836054, 0.5661...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
219,"[None, None, None, None, None, None, None, Non...",MDELRERNIVLAKLCEQAERYDEMVKAMIEIATNTETELTVEERNL...,"[[-0.038774293, -0.18798219, -0.061563067, 0.2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [31]:
n = 0

example_input = torch.tensor(np.stack(data["sequence_oh"].values[n:n + 1]), dtype=torch.float32)
example_output = torch.tensor(np.stack(data["pfams_indices"].values[n:n + 1]), dtype=torch.float32)

output = model_oh(example_input)

# print all values in the output tensor into console
output = output.view(-1, output.shape[-1])  # flatten for easier viewing
output = output.argmax(axis=-1)  # get the predicted class indices

print(example_output.shape)
print(example_output)

# Convert output to numpy array for easier viewing
example_output = example_output.view(-1, example_output.shape[-1])  # flatten for easier viewing
example_output = example_output.argmax(axis=-1)  # get the true class indices

print()
print([int(i) for i in output.numpy()])
print([int(i) for i in example_output.numpy()])

torch.Size([1, 1000])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
         2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
      

In [86]:
gt_list=[]
prediction_list=[]
for n in range(0,100):
    example_input = torch.tensor(np.stack(data["embedding"].values[n:n + 1]), dtype=torch.float32)
    example_output = torch.tensor(np.stack(data["pfams_indices"].values[n]), dtype=torch.float32)

    output = model_emb(example_input)
    output = output.view(-1, output.shape[-1])  # flatten for easier viewing
    output = output.argmax(axis=-1)
    gt_list.append(example_output.numpy())
    prediction_list.append(output.numpy())

# print all values in the output tensor into console
output = output.view(-1, output.shape[-1])  # flatten for easier viewing
output = output.argmax(axis=-1)  # get the predicted class indices

#print(output.shape)
#print(output)

# Convert output to numpy array for easier viewing
example_output = example_output.view(-1, example_output.shape[-1])  # flatten for easier viewing
example_output = example_output.argmax(axis=-1)  # get the true class indices
#print(example_output.shape)
#print(example_output)

print(data["pfams_indices"].values[n:n + 1])
print("----------------------------")
print(f"pred\n{[int(i) for i in output.numpy()]}")
print(f"gt\n{[int(i) for i in example_output.numpy()]}")

[array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [88]:
import pandas as pd

df = pd.DataFrame({"prediction": prediction_list, "gt": gt_list})
df

,prediction,gt
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, ..."
3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
...,...,...
95,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
96,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
97,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
98,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."


In [89]:
df.to_pickle("../dataset/example_results100.pkl")

In [90]:
df2 = pd.read_pickle("../dataset/example_results100.pkl")

In [92]:
df2["prediction"][10]

array([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,  54,  46, 128, 128, 128, 128, 170,
       128, 114, 128, 170,   1, 170,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1, 173,   1,
         1,   1,  46,   1,   1,   1,  46,   1,   1,   1,   1,   1,   1,
         1,  46,   1,  19,   1,  83,   1,  49,   1,  64,   1,  19,   1,
        28,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   

In [93]:
from collections import defaultdict

pfam_tp_dict = defaultdict(int)
pfam_fp_dict = defaultdict(int)
pfam_fn_dict = defaultdict(int)
n=0
for index, row in df2.iterrows():
    for idx in range(row["prediction"].shape[0]):
        if row["prediction"][idx] == row["gt"][idx]:
            pfam_tp_dict[row["gt"][idx]] += 1
        else:
            pfam_fn_dict[row["gt"][idx]] += 1
            pfam_fp_dict[row["gt"][idx]] += 1
        n+=1



In [94]:
pfams=[]
precision_list=[]
recall_list=[]
false_pos_list=[]
for pfam in pfam_tp_dict:
    tp = pfam_tp_dict[pfam]
    fp = pfam_fp_dict[pfam]
    fn = pfam_fn_dict[pfam]
    tn = n-(tp+fp+fn)
    precision = tp / (tp + fp)
    recall= tp/(tp + fn)
    fpr = fp / (fp + tn)
    precision_list.append(precision)
    recall_list.append(recall)
    false_pos_list.append(fpr)
    pfams.append(pfam)

scores_df=pd.DataFrame({"pfams":pfams, "precision":precision_list, "recall":recall_list,"false postive rate":false_pos_list})
scores_df

,pfams,precision,recall,false postive rate
0,1.0,0.975595,0.975595,0.001781
1,0.0,0.998291,0.998291,0.004519
2,3.0,0.107143,0.107143,0.006294
3,5.0,0.003203,0.003203,0.012607
4,2.0,0.001962,0.001962,0.026095
5,7.0,0.029499,0.029499,0.003301
6,8.0,0.005352,0.005352,0.013182


# 0 is padding and 1 is no pfam present

In [100]:
df2

,prediction,gt
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, ..."
3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
...,...,...
95,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
96,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
97,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
98,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."


In [106]:
test_row["prediction"]

array([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1, 104,   1, 104,   1, 104,   1, 104,   1,
       104,   1,   6,   1,   6,  21, 104,  28, 135,  21, 135,  21, 135,
         1, 133,   1, 133,   1,   1,   1, 128,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,  59,
        28,  59,  28,  59,  28, 132,  28, 126,  28, 126,  59, 126, 162,
         0,   0,   0,  59,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0, 133, 128,   1, 128,   1, 128, 128, 128, 128, 128, 128, 128,
       128, 128,  29, 162, 162, 128, 162, 162, 162,   1, 162,   

In [107]:
test_row = df2.iloc[0]
for idx in range(test_row["prediction"].shape[0]):
    print(test_row["prediction"][idx], test_row["gt"][idx])

1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 2.0
1 2.0
104 2.0
1 2.0
104 2.0
1 2.0
104 2.0
1 2.0
104 2.0
1 2.0
104 2.0
1 2.0
6 2.0
1 2.0
6 2.0
21 2.0
104 2.0
28 2.0
135 2.0
21 2.0
135 2.0
21 2.0
135 2.0
1 2.0
133 2.0
1 2.0
133 2.0
1 2.0
1 2.0
1 2.0
128 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
1 2.0
59 2.0
28 2.0
59 2.0
28 2.0
59 2.0
28 2.0
132 2.0
28 2.0
126 2.0
28 2.0
126 2.0
59 2.0
126 2.0
162 2.0
0 2.0
0 2.0
0 2.0
59 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
0 2.0
133 2.0
12

In [109]:
scores_df.to_pickle("test_sequences_scores.pkl")